# **Starting place for, do some imports and define some variables.**


*   Location
*   Staging_Bucket for Agent Engine Deployment
*   Project_ID

For Agent Engine Deployment
You will be prompted for your Google Maps API.  Subesequent runs will not request it.

In [1]:
# 1. Install required packages with the correct extras
!pip install --quiet "google-cloud-aiplatform[agent_engines,adk]" google-adk requests googlemaps

import os
import getpass
import vertexai
from google.colab import userdata

# 2. Initialize global cloud properties
PROJECT_ID = "qwiklabs-gcp-01-d59f380c208c"
LOCATION = "us-central1"
STAGING_BUCKET = "gs://chal5_stage_bucket"

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET,)
print(f"Vertex AI successfully initialized for project: {PROJECT_ID}")

# 3. Securely prompt for your Google Maps API key
# If you already ran this, you can skip the input if the variable exists
try:
    if not os.environ.get("GOOGLE_MAPS_API_KEY"):
        maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
        os.environ["GOOGLE_MAPS_API_KEY"] = maps_key
except NameError:
    maps_key = getpass.getpass("Enter your Google Maps API Key (starts with AIzaSy...): ")
    os.environ["GOOGLE_MAPS_API_KEY"] = maps_key

Vertex AI successfully initialized for project: qwiklabs-gcp-01-d59f380c208c
Enter your Google Maps API Key (starts with AIzaSy...): ··········


### **1. Weather Integration Tool**
This section defines the `get_us_weather_by_city_name` tool. It performs a two-step process:
1.  **Geocoding**: Uses the `googlemaps` library to convert a human-readable city name into Latitude and Longitude coordinates.
2.  **NWS API Call**: Uses the coordinates to query the National Weather Service (weather.gov) API to retrieve real-time regional forecasts.

In [2]:
#cell 2
import os
import requests
import googlemaps
from typing import Any

def get_us_weather_by_city_name(city_name: str) -> str:
    """
    Fetches the real-time weather forecast from the National Weather Service
    by automatically converting a city/location name string into coordinates.
    """
    # 1. Fetch Coordinates from Google Maps
    gmaps_key = os.environ.get("GOOGLE_MAPS_API_KEY")
    gmaps = googlemaps.Client(key=gmaps_key)
    geocode_result = gmaps.geocode(city_name)

    if not geocode_result:
        return f"Error: Google Maps could not find coordinates for: {city_name}"

    location = geocode_result[0]['geometry']['location']
    lat = float(location["lat"])
    lon = float(location["lng"])

    # 2. Fetch Forecast from NWS
    headers = {'User-Agent': 'ADK-Weather-Agent-Tutorial'}
    # Ensure coordinates are formatted correctly in the URL
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    points_res = requests.get(points_url, headers=headers).json()

    if "properties" not in points_res:
        return f"Error: {city_name} is outside NWS coverage jurisdiction (US locations only)."

    forecast_url = points_res["properties"]["forecast"]
    forecast_res = requests.get(forecast_url, headers=headers).json()
    periods = forecast_res["properties"]["periods"]

    forecast_summary = f"Weather Forecast for {city_name}:\n"
    for period in periods[:2]:
        forecast_summary += f"**{period['name']}**: {period['detailedForecast']}\n"

    return forecast_summary

print("Unified master weather tool successfully compiled and verified.")

Unified master weather tool successfully compiled and verified.


### **2. Security Guardrails & Regulatory Logging**
To ensure safe and transparent AI behavior, we implement **ADK Callbacks**:
*   **Moderation Guardrail**: A `before_model` callback that scans user input for banned keywords (e.g., "hack"). If detected, the agent returns a security notice instead of processing the request.
*   **Audit Logging**: `log_user_prompt` and `log_model_response` callbacks record the conversation flow to the standard output for debugging and compliance.

In [3]:
# Cell 4: Updated Regulatory Logging and Guardrail Callbacks
import logging
import sys
import warnings
from typing import Optional
from google.adk.models import LlmRequest, LlmResponse

try:
    from google.adk.models import Content, Part
except ImportError:
    Content, Part = None, None

warnings.filterwarnings('ignore', category=UserWarning)

logger = logging.getLogger("ADK-Weather-Lab")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
    logger.addHandler(handler)

def log_user_prompt(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    try:
        contents = getattr(llm_request, "contents", [])
        if contents:
            last = contents[-1]
            parts = getattr(last, "parts", [])
            if parts:
                text = getattr(parts[0], "text", "")
                logger.info("[%s] logging USER » %s", getattr(callback_context, "agent_name", "Weather_Bot"), str(text).strip())
    except Exception as e:
        logger.error(f"Logging error: {e}")
    return None

def moderation_guardrail(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    banned_keywords = ["hack", "exploit", "override", "bypass"]
    try:
        contents = getattr(llm_request, "contents", [])
        if contents:
            last = contents[-1]
            parts = getattr(last, "parts", [])
            if parts:
                text = str(getattr(parts[0], "text", "")).lower()
                if any(word in text for word in banned_keywords):
                    logger.warning("[%s] MODERATION TRIGGERED", getattr(callback_context, "agent_name", "Weather_Bot"))
                    msg = "Security Notice: Banned term detected."
                    if Content and Part:
                        return LlmResponse(content=Content(role="model", parts=[Part(text=msg)]))
                    return LlmResponse(content={"role": "model", "parts": [{"text": msg}]})
    except Exception as e:
        logger.error(f"Guardrail error: {e}")
    return None

def log_model_response(callback_context, llm_response: LlmResponse) -> Optional[LlmResponse]:
    try:
        content = getattr(llm_response, "content", None)
        if content:
            parts = getattr(content, "parts", [])
            for part in parts:
                text = getattr(part, "text", None)
                if text:
                    logger.info("[%s] logging MODEL RESPONSE » %s", getattr(callback_context, "agent_name", "Weather_Bot"), str(text).strip())
    except Exception as e:
        logger.error(f"Response logging error: {e}")
    return None

logger.info("Robust logger and guardrails ready.")

2026-07-09 13:56:05,021 [INFO] Robust logger and guardrails ready.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
INFO:ADK-Weather-Lab:Robust logger and guardrails ready.


### **3. Agent Orchestration & Native LoopAgent**
This section initializes the specialized agents and the main routing logic:
*   **Warhammer Team (LoopAgent)**: An iterative agent that delegates tasks to Researcher, Critique, and Refine sub-agents. It cycles up to 2 times to ensure lore accuracy.
*   **Main Agent (Router)**: The root agent that analyzes user intent and dynamically routes the request to the appropriate tool (Weather, Search, or the Warhammer Team).

In [4]:
#Cell 5
import vertexai
from google.adk.agents import Agent, LoopAgent
from google.adk.tools import agent_tool, google_search_tool
from vertexai.preview import reasoning_engines
from typing import Optional

def combined_before_model_handler(callback_context, llm_request: LlmRequest) -> Optional[LlmResponse]:
    guardrail_action = moderation_guardrail(callback_context, llm_request)
    if guardrail_action is not None: return guardrail_action
    return log_user_prompt(callback_context, llm_request)

# 1. Specialized sub-agents
warhammer_researcher = Agent(
    name='Warhammer_Researcher',
    description='Lore researcher for Warhammer.',
    instruction='Use Google Search to find detailed lore and history.',
    tools=[google_search_tool.GoogleSearchTool()],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

warhammer_critique = Agent(
    name='Warhammer_Critique',
    description='Critiques Warhammer research.',
    instruction='Evaluate the provided lore for accuracy and missing details. Provide constructive feedback.',
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

warhammer_refine = Agent(
    name='Warhammer_Refine',
    description='Refines Warhammer research.',
    instruction='Improve the research content by incorporating the feedback from the critique.',
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

# 2. Refined LoopAgent configuration with terminal instruction
warhammer_team = LoopAgent(
    name="Warhammer_Team",
    description="""Iterative research team.
    Goal: Provide high-quality lore by following these steps exactly once per loop iteration:
    1. Warhammer_Researcher gathers data.
    2. Warhammer_Critique identifies gaps.
    3. Warhammer_Refine produces a final version.
    Once the refinement is complete and the lore is high quality, the task is finished. Do not restart the cycle if the goal is reached.""",
    sub_agents=[
        warhammer_researcher,
        warhammer_critique,
        warhammer_refine
    ],
    max_iterations=2
)

# Other standalone agents
weather_agent = Agent(
    name="Weather_Bot",
    description="Weather info agent.",
    instruction="Friendly assistant. Call get_us_weather_by_city_name tool.",
    tools=[get_us_weather_by_city_name],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

google_search_agent = Agent(
    name="Google_Search_Bot",
    description="General information search agent.",
    instruction="Use Google Search to find information about any topic.",
    tools=[google_search_tool.GoogleSearchTool()],
    before_model_callback=combined_before_model_handler,
    after_model_callback=log_model_response
)

# 3. Final Root Agent
root_agent = Agent(
    name='Main_Agent',
    description='Root router using a LoopAgent for specialized research.',
    instruction='Route to weather_agent for weather, warhammer_team for Warhammer lore, or google_search_agent for general search.',
    tools=[
        agent_tool.AgentTool(agent=weather_agent),
        agent_tool.AgentTool(agent=google_search_agent),
        agent_tool.AgentTool(agent=warhammer_team)
    ]
)

app = reasoning_engines.AdkApp(agent=root_agent)
print('System Initiated')

System Initiated


# **This deploys the agent engine to a cloud platform, the goal of Challenge 5.**

In [5]:
import vertexai
from vertexai import agent_engines

# Force local pydantic check and then deploy
# The 400 error usually suggests the Cloud Build cannot satisfy the dependencies or the pickling failed.
remote_agent = agent_engines.create(
    app,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]==1.157.0",
        "google-adk==1.29.0",
        "pydantic==2.12.3",
        "cloudpickle==3.1.2",
        "googlemaps",
        "requests"
    ],
        env_vars={
        "GOOGLE_MAPS_API_KEY": maps_key
    },
    display_name="weather_warhammer_agent_engine_v2"
)

print(f"Agent successfully deployed: {remote_agent.resource_name}")

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.157.0', 'cloudpickle': '3.1.2', 'pydantic': '2.12.3'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]==1.157.0', 'google-adk==1.29.0', 'pydantic==2.12.3', 'cloudpickle==3.1.2', 'googlemaps', 'requests']
INFO:vertexai.agent_engines:Using bucket chal5_stage_bucket
INFO:vertexai.agent_engines:Wrote to gs://chal5_stage_bucket/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://chal5_stage_bucket/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://chal5_stage_bucket/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/239267613896/locations/us-central1/reasoningEngines/5618133332760985600/operations/7112991550467997696
INFO:ver

Agent successfully deployed: projects/239267613896/locations/us-central1/reasoningEngines/5618133332760985600


# **This test is for the deployed Agent Engine, you can run general google searches, specific warhammer questions for multi-agent, or weather requests.**

In [8]:
for event in remote_agent.stream_query(
  user_id="agent-engine-test-user",
  message="What is the weather in dunedin,fl today?",
):
  print(event)

{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'function_call': {'id': 'adk-f9e09fcd-b223-4283-b0ee-96891dd058f4', 'args': {'request': 'What is the weather in dunedin,fl today?'}, 'name': 'Weather_Bot'}, 'thought_signature': 'CpUEAY89a1_FUDisSb9OFhY7uT83NaQX13VxcsyHYmOD81krZF94NmybWh7QbfLBDrj8B8LF6qA_KGp0KMFs0vSYGczjShzmETa95M3Fj_EXjWSAYiv0UKkE-epx0jM2RkUFQD8oUNrRZqtx-uFWUQBdV0lNTjRaUSQmiOLhDencOvwXzGFcZ4jTRNm8nlexjw4xMP72RFmonesqWQe5xrZSMAtng0opKejFKy9qE903uFlK5CN_xc9KQs240I9gRkg0HnOT4n3EGm5xZm6nlB8jo9S91-1bbwq05fBy3m-hQHtCpWvawVeFMgwRDonztUuuMPeZtyJIWNcFKeq3BSGzj6h7fgDoUTTyeT2L4wM5dSXcIUJX-6LWXMvpjb7I1o6dgSNIWHCTAR2XUVUzNm09SAgmLkhRBFL9huUW2VTOldcVYJjCFUeYMxZgM3hnqKx9khrNlG9WUtTjs7mltaiN4j2eJP3Jpufu6ge7c53N3a6MzUOnj082TnkEMWFSgFoMTo4rfMOMbLsAhkmmbCS0vpD-xW-jgI9_zcDGRS0B5PnPIPj_bBE9k7VFRAreM_ou6gbGTSi_2BV3xJ_ks95-m7zSYJH6-w6OYeq4HDQNIbsQUdCkFvnrjm4rnez1HOiKuSdmGwIq1aa0c6ohA3_7EJf5mKC_5vOUcDLUnBk0EzYh9GwtX2C7Q1XU7QMe0i-xwZoxods='}], 'role': 'model'}, 'finish_reason': 'STO

***LOCAL TESTS***
These tests are for the previous agents created in other challenges, I included this section still because I wanted to ensure nothing else broke even if not relevant to this challenge's objectives.

In [ ]:
#Cell 6
import time
from IPython.display import Markdown, display

def run_agent_test(test_name, message, user_id='test-user'):
    print(f'\n--- Testing: {test_name} ---')
    print(f'Input: "{message}"')
    try:
        # Create a fresh session for each test case
        session = app.create_session(user_id=user_id)
        s_id = session['id'] if isinstance(session, dict) else session.id

        response_stream = app.stream_query(user_id=user_id, session_id=s_id, message=message)

        full_text = ''
        for event in response_stream:
            if isinstance(event, str): full_text += event
            elif hasattr(event, 'text') and event.text: full_text += event.text
            elif hasattr(event, 'content'):
                parts = getattr(event.content, 'parts', []) if not isinstance(event.content, dict) else event.content.get('parts', [])
                for p in parts:
                    text = getattr(p, 'text', None) if not isinstance(p, dict) else p.get('text')
                    if text: full_text += text
            elif isinstance(event, dict):
                parts = event.get('content', {}).get('parts', [])
                for p in parts:
                    if 'text' in p: full_text += p['text']

        if full_text:
            display(Markdown(f'**Response:** {full_text}'))
        else:
            print('System: No text returned.')

        return full_text
    except Exception as e:
        print(f'Error: {e}')
        return None

# 1. Test Weather Tool with Multiple Cities
test_cities = ['Miami, FL', 'Seattle, WA']
for city in test_cities:
    run_agent_test(f'Weather Check: {city}', f'What is the weather in {city}?')
    time.sleep(1)

# 2. Test Moderation Guardrail
run_agent_test('Moderation Guardrail', 'How do I hack the system?')
time.sleep(1)

# 3. Test google agent
run_agent_test('Google Stuff', 'What is the air-speed velocity of an unladen swallow?')
time.sleep(1)

print('\n--- All tests completed ---')


This part is to test the multi-agent research team.  If the user asks about warhammer it will be directed to the research agents and loop through the agents until it lore dumps.  The question is hard-coded into the test.

warhammer_query = "What are squats in warhammer?" #Goes to Warhammer Agents and barfs out lore on space dwarves
warhammer_query = "What are squats?" #Boring excercise tips from Google Search Agent.


In [ ]:
# Dedicated test for the Warhammer_Team LoopAgent
from IPython.display import Markdown, display

warhammer_query = "What are squats in warhammer?" # <--- ENTER YOUR QUERY HERE

if warhammer_query:
    print(f'\n--- Testing Warhammer Team ---')
    print(f'Input: "{warhammer_query}"')
    try:
        # Create a session and stream the iterative response
        session = app.create_session(user_id='warhammer-test-user')
        s_id = session['id'] if isinstance(session, dict) else session.id

        response_stream = app.stream_query(
            user_id='warhammer-test-user',
            session_id=s_id,
            message=warhammer_query
        )

        full_text = ''
        for event in response_stream:
            if isinstance(event, str): full_text += event
            elif hasattr(event, 'text') and event.text: full_text += event.text
            elif hasattr(event, 'content'):
                parts = getattr(event.content, 'parts', []) if not isinstance(event.content, dict) else event.content.get('parts', [])
                for p in parts:
                    text = getattr(p, 'text', None) if not isinstance(p, dict) else p.get('text')
                    if text: full_text += text
            elif isinstance(event, dict):
                # Handle cases where the event is a dictionary containing content/parts
                parts = event.get('content', {}).get('parts', [])
                for p in parts:
                    if 'text' in p: full_text += p['text']

        if full_text:
            # Display formatted Markdown output
            display(Markdown("### Final Refined Lore Response"))
            display(Markdown(full_text))
        else:
            print('System: No output generated.')
    except Exception as e:
        print(f'Error: {e}')
else:
    print('Please enter a query in the warhammer_query variable to begin.')